# Model Analysis

In this notebook we will analyse the robustness of this model against typical attacks. We will explore whether or not the use of uncertainty can make the model aware of slight changes in the input.

## Import libraries

In [1]:
import torch
import json
import matplotlib.pyplot as plt
import numpy as np
import tqdm
from collections import defaultdict

In [2]:
from src.data import load_and_prepare_datasets
from src.utils import load_model
from src.gaussian_processes.prediction import predict_with_uncertainty

c:\Master\PAI\uncertainty-aware-moderation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
model, tokenizer = load_model("models/gp_best", device=device)
model.eval()

[INFO] Loading model from: models/gp_best


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10855.67it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Model loaded successfully.


DistilBERTWithGP(
  (encoder): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Li

## Data exploration

In [5]:
datasets, id2label = load_and_prepare_datasets(
    tokenizer=tokenizer,
)
 
test_set = datasets["test"]
print("longitud test set:", len(test_set))

Jigsaw dataset already exists. Skipping download.


Map: 100%|██████████| 15958/15958 [00:01<00:00, 15767.33 examples/s]

longitud test set: 15958


In [6]:
i = 4
sample = test_set[i]
 
text = tokenizer.decode(sample["input_ids"], skip_special_tokens=True)
label = sample["labels"]

print("======== EJEMPLO =========")
 
print("TEXT:", text)
print("TRUE LABELS:", label.numpy())
print("CLASS NAMES:", id2label)

print("\nPREDICCIONES:")

result = predict_with_uncertainty(model, tokenizer, text, id2label, device="cuda")
for label, prob, std in result["all_scores"]:
        print(f"{label:15s} | prob={prob:.4f} | std={std:.4f}")

print("=========================")

======== EJEMPLO =========
TEXT: fuck you wanker
TRUE LABELS: [1. 1. 1. 0. 1. 0.]
CLASS NAMES: ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

PREDICCIONES:
toxic           | prob=0.9948 | std=0.0717
severe_toxic    | prob=0.4348 | std=0.4957
obscene         | prob=0.9919 | std=0.0895
threat          | prob=0.0080 | std=0.0890
insult          | prob=0.8691 | std=0.3373
identity_hate   | prob=0.1451 | std=0.3522


### Agrupar samples por combinación y guardar incertidumbre

In [7]:
combos = defaultdict(list)

for i, sample in tqdm.tqdm(enumerate(test_set)):
    labels = tuple(int(x) for x in sample["labels"].numpy())
    text = tokenizer.decode(sample["input_ids"], skip_special_tokens=True)

    result = predict_with_uncertainty(
        model=model,
        tokenizer=tokenizer,
        text=text,
        id2label=id2label,
        device=device,
        threshold=0.5,
        top_k=3,
    )

    # probs y stds por label
    probs = np.array([p for _, p, _ in result["all_scores"]])
    stds  = np.array([s for _, _, s in result["all_scores"]])

    pred_labels = (probs >= 0.5).astype(int)
    correct = np.array_equal(np.array(labels), pred_labels)

    # métrica global de incertidumbre
    uncertainty = float(stds.mean())   # opción recomendada

    combos[labels].append({
        "index": i,
        "text": text,
        "combo_true": labels,
        "combo_pred": tuple(pred_labels.tolist()),
        "correct": correct,
        "uncertainty": uncertainty,
        "probs": probs.tolist(),
        "stds": stds.tolist(),
    })

15958it [06:07, 43.47it/s]


### Quedarte con los 2 más inciertos por combinación

In [8]:
top2_uncertain_per_combo = {}

for combo, samples in combos.items():
    top2 = sorted(samples, key=lambda x: x["uncertainty"], reverse=True)[:2]
    top2_uncertain_per_combo[combo] = top2

serializable = {
    str(combo): samples
    for combo, samples in top2_uncertain_per_combo.items()
}

# serializable = [
#     {
#         "combo": list(combo),
#         "samples": samples
#     }
#     for combo, samples in top2_uncertain_per_combo.items()
# ]

with open("top2_uncertain_per_combo.json", "w", encoding="utf-8") as f:
    json.dump(serializable, f, ensure_ascii=False, indent=2)

### Mostrar resultados

In [10]:
for combo in sorted(top2_uncertain_per_combo.keys()):
    print("\n==========================================")
    print("COMBO:", combo)

    samples = top2_uncertain_per_combo[combo]

    if not samples:
        print("No hay samples")
        continue

    for rank, s in enumerate(samples, 1):
        print(f"\n#{rank} más incierto — idx {s['index']}")
        print("TRUE:", s["combo_true"])
        print("PRED:", s["combo_pred"])
        print("CORRECT:", s["correct"])
        print(f"UNCERTAINTY(mean std): {s['uncertainty']:.4f}")
        print("STDS:", np.round(s["stds"], 4))
        print("PROBS:", np.round(s["probs"], 4))
        print("TEXT:", s["text"])


COMBO: (0, 0, 0, 0, 0, 0)

#1 más incierto — idx 13675
TRUE: (0, 0, 0, 0, 0, 0)
PRED: (1, 0, 0, 0, 0, 0)
CORRECT: False
UNCERTAINTY(mean std): 0.4124
STDS: [0.4682 0.3929 0.5    0.1907 0.4761 0.4467]
PROBS: [0.6753 0.1907 0.4937 0.0378 0.3471 0.2753]
TEXT: c o b a l t b l u e t o n y = eunuch

#2 más incierto — idx 2915
TRUE: (0, 0, 0, 0, 0, 0)
PRED: (1, 0, 0, 0, 0, 1)
CORRECT: False
UNCERTAINTY(mean std): 0.3653
STDS: [0.4452 0.2019 0.4714 0.1496 0.4549 0.4685]
PROBS: [0.7276 0.0426 0.3332 0.0229 0.2925 0.6746]
TEXT: racists i think every hick member of the national alliance should read guns, germs and steel. then, white supremacy would cease to exist. seriously.

COMBO: (0, 0, 0, 0, 0, 1)

#1 más incierto — idx 12148
TRUE: (0, 0, 0, 0, 0, 1)
PRED: (0, 0, 0, 0, 0, 0)
CORRECT: False
UNCERTAINTY(mean std): 0.2818
STDS: [0.495  0.0873 0.3518 0.0388 0.2867 0.4314]
PROBS: [0.4293 0.0077 0.1447 0.0015 0.0904 0.2472]
TEXT: it is obvius to, me, a black man, that orange is for da black gs, bl

### ¿El modelo falla más cuando es más incierto?

In [12]:
all_samples = []
for combo_list in combos.values():
    all_samples.extend(combo_list)

uncertainties = np.array([x["uncertainty"] for x in all_samples])
corrects = np.array([x["correct"] for x in all_samples]).astype(int)

# ordenar por incertidumbre
order = np.argsort(uncertainties)
unc_sorted = uncertainties[order]
corr_sorted = corrects[order]

# dividir en bins (ej: 10)
n_bins = 10
bins = np.array_split(np.arange(len(unc_sorted)), n_bins)

print("\nAccuracy por nivel de incertidumbre:")
for i, b in enumerate(bins):
    acc = corr_sorted[b].mean()
    unc_mean = unc_sorted[b].mean()
    print(f"Bin {i}: unc={unc_mean:.3f} | acc={acc:.3f}")


Accuracy por nivel de incertidumbre:
Bin 0: unc=0.012 | acc=0.999
Bin 1: unc=0.016 | acc=0.999
Bin 2: unc=0.020 | acc=1.000
Bin 3: unc=0.024 | acc=0.996
Bin 4: unc=0.030 | acc=0.998
Bin 5: unc=0.039 | acc=0.991
Bin 6: unc=0.052 | acc=0.987
Bin 7: unc=0.077 | acc=0.953
Bin 8: unc=0.138 | acc=0.818
Bin 9: unc=0.268 | acc=0.415


### Correlación incertidumbre-error

In [13]:
# error = 1 si falla
errors = 1 - corrects

corr = np.corrcoef(uncertainties, errors)[0, 1]
print(f"Correlación incertidumbre-error: {corr:.3f}")

Correlación incertidumbre-error: 0.635


### Acc total

In [14]:
accuracy = corrects.mean()
print(f"Accuracy total: {accuracy:.4f}")

Accuracy total: 0.9157


### Acc por label

In [15]:
num_labels = len(all_samples[0]["probs"])

y_true = []
y_pred = []

for x in all_samples:
    y_true.append(np.array(x["combo_true"]))
    y_pred.append(np.array(x["combo_pred"]))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\nAccuracy por label:")
for i in range(num_labels):
    acc = (y_true[:, i] == y_pred[:, i]).mean()
    print(f"Label {i}: {acc:.4f}")


Accuracy por label:
Label 0: 0.9543
Label 1: 0.9907
Label 2: 0.9749
Label 3: 0.9977
Label 4: 0.9688
Label 5: 0.9924


### F1 por label

In [16]:
from sklearn.metrics import f1_score

print("\nF1 por label:")
for i in range(num_labels):
    f1 = f1_score(y_true[:, i], y_pred[:, i])
    print(f"Label {i}: {f1:.4f}")


F1 por label:
Label 0: 0.7075
Label 1: 0.2437
Label 2: 0.7315
Label 3: 0.0000
Label 4: 0.5958
Label 5: 0.3086


### Acc en top-k más inciertos

In [17]:
k = int(0.2 * len(all_samples))  # top 20% más inciertos

top_idx = np.argsort(uncertainties)[-k:]
low_idx = np.argsort(uncertainties)[:k]

acc_high_unc = corrects[top_idx].mean()
acc_low_unc = corrects[low_idx].mean()

print(f"\nTop 20% más inciertos → acc: {acc_high_unc:.3f}")
print(f"Top 20% más seguros → acc: {acc_low_unc:.3f}")


Top 20% más inciertos → acc: 0.617
Top 20% más seguros → acc: 0.999


How to Read These F1-macro and F1-micro?
 
The metrics we got are completely **normal and healthy**, showing the model learned effectively without overfitting too much.
 
* The F1-micro score (around **0.79**) is strong and shows good **generalization**, it's performing well on new data it's never seen. The small difference between the training F1-micro (~0.87) and the validation F1-micro (~0.79) is good; it means the model is **stable** and not just memorizing the training examples.
 
* The F1-macro score (around **0.65**) is lower, but that is **expected** for the Jigsaw dataset. The F1-macro score is pulled down by the **imbalance** in the data, where super rare labels (like *threat* or *identity\_hate*) get low individual scores.
 
Here's how these two scores are calculated:
 
* **F1-micro:** It calculates the F1 score globally, counting up the total correct and incorrect predictions across all labels. It basically treats all predictions as a single, combined pool.
 
    $$
    \text{F1}_{\text{micro}} =
    \frac{2 \cdot \text{TP}}
        {\,2 \cdot \text{TP} + \text{FP} + \text{FN}\,}
    $$
 
 
* **F1-macro:** It calculates the F1 score for *each* of the six toxicity labels separately, and then takes a simple average of those six scores.
 
    $$
    \text{F1}_{\text{macro}} =
    \frac{1}{6}
    \sum_{i=1}^{6} \text{F1}_i
    $$